# Mini laboratorio · ECE 2026
## Métricas para evaluar el desempeño de un algoritmo multiobjetivo

**Objetivo.** Pasar de los conjuntos sintéticos del taller a una comparación real entre **NSGA-II** y **SMS-EMOA** sobre **ZDT1**.

Usaremos el **mismo presupuesto de evaluaciones** y compararemos:

- **Hypervolume (HV)** — mayor es mejor.
- **IGD+** — menor es mejor.
- **$\epsilon^+$ aditivo** — menor es mejor.

También veremos **cómo evolucionan HV e IGD+ durante la búsqueda**, no sólo el resultado final.

> La profesora puede ejecutar el notebook en pantalla. Quienes tengan computadora pueden correrlo al mismo tiempo; quienes no, participan haciendo predicciones e interpretando los resultados.

Material del taller: https://margaritareyess.github.io/ece2026/


> **Versión 2 del mini laboratorio.**  
> Incluye una comparación explícita entre **1200 y 2000 evaluaciones** para mostrar que la interpretación también puede depender del presupuesto de búsqueda.


### Antes de ejecutar: haz una predicción

1. ¿Cuál algoritmo crees que terminará con mayor HV?
2. ¿Cuál crees que terminará con menor IGD+?
3. ¿Esperas que ambos indicadores cuenten exactamente la misma historia?

**No buscamos adivinar correctamente.** Queremos contrastar la intuición con una ejecución real.


In [ ]:
# Ejecutar una sola vez en Google Colab
!pip -q install pymoo


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.sms import SMSEMOA
from pymoo.core.callback import Callback
from pymoo.indicators.hv import HV
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.optimize import minimize
from pymoo.problems import get_problem

print("Entorno listo.")


## 1. Configuración del experimento

Usamos **ZDT1**, el mismo tipo de frente del taller. Ambos algoritmos reciben el mismo tamaño de población, la misma semilla y el mismo presupuesto total de evaluaciones.

> SMS-EMOA utiliza contribución de hypervolume durante la selección ambiental; NSGA-II usa ordenamiento no dominado y crowding distance.


In [ ]:
SEED = 7
POP_SIZE = 50
BUDGET = 2000          # Para una demo todavía más rápida: prueba 1200
CHECKPOINT = 100

problem = get_problem("zdt1")

# Pareto front analítico y denso de ZDT1
f1 = np.linspace(0.0, 1.0, 1001)
PF = np.column_stack([f1, 1.0 - np.sqrt(f1)])

REF_POINT = np.array([1.1, 1.1])

hv_indicator = HV(ref_point=REF_POINT)
igdplus_indicator = IGDPlus(PF)

print(f"Población = {POP_SIZE}")
print(f"Presupuesto = {BUDGET} evaluaciones")
print(f"Semilla = {SEED}")
print(f"Reference point HV = {REF_POINT}")


## 2. Funciones auxiliares

El *callback* registra únicamente los indicadores y algunos approximation sets durante la ejecución. También calcularemos $\epsilon^+$ aditivo al final.


In [ ]:
def additive_epsilon(A, R):
    """Indicador epsilon aditivo unary respecto al reference set R. Menor es mejor."""
    A = np.asarray(A, dtype=float)
    R = np.asarray(R, dtype=float)

    best_for_each_r = []
    for r in R:
        eps_candidates = np.max(A - r, axis=1)
        best_for_each_r.append(np.min(eps_candidates))

    return float(np.max(best_for_each_r))


class MetricCallback(Callback):
    def __init__(self, hv, igdplus, checkpoint=100):
        super().__init__()
        self.hv = hv
        self.igdplus = igdplus
        self.checkpoint = checkpoint
        self.next_checkpoint = 0

        self.data["n_eval"] = []
        self.data["HV"] = []
        self.data["IGD+"] = []
        self.data["F"] = []

    def notify(self, algorithm):
        n_eval = algorithm.evaluator.n_eval

        if len(self.data["n_eval"]) == 0 or n_eval >= self.next_checkpoint:
            F = algorithm.opt.get("F").copy()

            self.data["n_eval"].append(n_eval)
            self.data["HV"].append(float(self.hv(F)))
            self.data["IGD+"].append(float(self.igdplus(F)))
            self.data["F"].append(F)

            self.next_checkpoint = max(
                self.checkpoint,
                ((n_eval // self.checkpoint) + 1) * self.checkpoint
            )


def run_algorithm(name, algorithm):
    callback = MetricCallback(
        hv=HV(ref_point=REF_POINT),
        igdplus=IGDPlus(PF),
        checkpoint=CHECKPOINT
    )

    t0 = time.perf_counter()

    res = minimize(
        problem,
        algorithm,
        termination=("n_eval", BUDGET),
        seed=SEED,
        callback=callback,
        verbose=False
    )

    elapsed = time.perf_counter() - t0

    hist = pd.DataFrame({
        "n_eval": res.algorithm.callback.data["n_eval"],
        "HV": res.algorithm.callback.data["HV"],
        "IGD+": res.algorithm.callback.data["IGD+"]
    })

    return {
        "name": name,
        "res": res,
        "history": hist,
        "snapshots": res.algorithm.callback.data["F"],
        "time": elapsed
    }

print("Funciones auxiliares listas.")


## 3. Ejecutamos los dos algoritmos

**Mismo problema + misma semilla + mismo presupuesto de evaluaciones.**

Antes de ejecutar, conviene pedir al grupo una última predicción.


In [ ]:
experiments = {}

experiments["NSGA-II"] = run_algorithm(
    "NSGA-II",
    NSGA2(pop_size=POP_SIZE)
)

experiments["SMS-EMOA"] = run_algorithm(
    "SMS-EMOA",
    SMSEMOA(pop_size=POP_SIZE)
)

for name, exp in experiments.items():
    print(
        f"{name:9s} | evaluaciones finales: "
        f"{exp['res'].algorithm.evaluator.n_eval:4d} | "
        f"tiempo: {exp['time']:.2f} s"
    )


## 4. Primera mirada: approximation fronts finales

Todavía **no miramos la tabla de métricas**. Primero observamos los conjuntos.

- ¿Cuál parece converger mejor?
- ¿Cuál cubre mejor los extremos?
- ¿Ves huecos o aglomeraciones?
- Si tuvieras que elegir visualmente, ¿cuál escogerías?


In [ ]:
plt.figure(figsize=(8, 6))

plt.plot(PF[:, 0], PF[:, 1], "--", linewidth=2, label="Pareto front")

for name, exp in experiments.items():
    F = exp["res"].F
    plt.scatter(F[:, 0], F[:, 1], s=34, label=name, alpha=0.85)

plt.xlabel("$f_1$")
plt.ylabel("$f_2$")
plt.title("Approximation fronts finales")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 5. Evolución del Hypervolume

Ahora observamos la búsqueda como un **proceso**.

En HV, **mayor es mejor**.


In [ ]:
plt.figure(figsize=(8, 5))

for name, exp in experiments.items():
    h = exp["history"]
    plt.plot(h["n_eval"], h["HV"], marker="o", markersize=3, label=name)

plt.xlabel("Número de evaluaciones")
plt.ylabel("Hypervolume")
plt.title("Evolución del Hypervolume")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 6. Evolución de IGD+

Ahora cambiamos la pregunta.

En IGD+, **menor es mejor**.


In [ ]:
plt.figure(figsize=(8, 5))

for name, exp in experiments.items():
    h = exp["history"]
    plt.plot(h["n_eval"], h["IGD+"], marker="o", markersize=3, label=name)

plt.xlabel("Número de evaluaciones")
plt.ylabel("IGD+")
plt.title("Evolución de IGD+")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 7. ¿Qué pasa si detenemos antes la búsqueda?

Las curvas sugieren que la historia puede cambiar con el **presupuesto de evaluaciones**.

Comparemos dos momentos:

- **1200 evaluaciones**: etapa intermedia.
- **2000 evaluaciones**: presupuesto final.

Para cada algoritmo calcularemos de nuevo **HV, IGD+ y $\epsilon^+$** usando el approximation set disponible en ese momento.


In [ ]:
def snapshot_at(exp, target_eval):
    """Devuelve el snapshot almacenado más cercano a target_eval."""
    h = exp["history"]
    idx = int(np.argmin(np.abs(h["n_eval"].to_numpy() - target_eval)))
    return int(h.iloc[idx]["n_eval"]), exp["snapshots"][idx]


def metrics_for_snapshot(F):
    return {
        "HV ↑": hv_indicator(F),
        "IGD+ ↓": igdplus_indicator(F),
        "epsilon+ ↓": additive_epsilon(F, PF),
        "n_solutions": len(F)
    }


BUDGETS_TO_COMPARE = [1200, BUDGET]

rows_budget = []

for target_eval in BUDGETS_TO_COMPARE:
    for name, exp in experiments.items():
        n_eval, F_snap = snapshot_at(exp, target_eval)
        metrics = metrics_for_snapshot(F_snap)

        rows_budget.append({
            "Presupuesto objetivo": target_eval,
            "Evaluaciones usadas": n_eval,
            "Algoritmo": name,
            **metrics
        })

budget_table = (
    pd.DataFrame(rows_budget)
      .set_index(["Presupuesto objetivo", "Algoritmo"])
)

budget_table.round(5)


### Lectura sugerida

Antes de interpretar la tabla:

> **¿El mismo algoritmo gana con el mismo indicador en ambos presupuestos?**

La idea no es sólo identificar un ganador, sino notar que una conclusión puede depender de **cuándo detenemos la búsqueda**.


In [ ]:
def best_algorithm(subtable, metric, maximize=False):
    values = subtable[metric]
    return values.idxmax() if maximize else values.idxmin()


winner_rows = []

for target_eval in BUDGETS_TO_COMPARE:
    sub = budget_table.loc[target_eval]

    winner_rows.append({
        "Presupuesto": target_eval,
        "Mejor HV": best_algorithm(sub, "HV ↑", maximize=True),
        "Mejor IGD+": best_algorithm(sub, "IGD+ ↓", maximize=False),
        "Mejor epsilon+": best_algorithm(sub, "epsilon+ ↓", maximize=False)
    })

winner_table = pd.DataFrame(winner_rows).set_index("Presupuesto")
winner_table


## 8. Frentes en dos momentos de la búsqueda

Ahora miramos **qué geometría produjo esas métricas**.

Para cada algoritmo mostramos el approximation set alrededor de:

- 1200 evaluaciones;
- 2000 evaluaciones.


In [ ]:
for name, exp in experiments.items():
    for target_eval in BUDGETS_TO_COMPARE:
        n_eval, F_snap = snapshot_at(exp, target_eval)

        plt.figure(figsize=(7, 5))
        plt.plot(PF[:, 0], PF[:, 1], "--", linewidth=2, label="Pareto front")
        plt.scatter(
            F_snap[:, 0], F_snap[:, 1],
            s=34,
            label=f"{name} @ {n_eval} evaluaciones"
        )
        plt.xlabel("$f_1$")
        plt.ylabel("$f_2$")
        plt.title(f"{name}: approximation set")
        plt.legend()
        plt.grid(alpha=0.25)
        plt.show()


## 9. Tabla final: tres maneras de mirar el mismo resultado

Sólo ahora revelamos los valores finales.

- **HV:** mayor es mejor.
- **IGD+:** menor es mejor.
- **$\epsilon^+$:** menor es mejor.

La pregunta importante no es únicamente **quién gana**, sino si las métricas cuentan la misma historia.


In [ ]:
rows = []

for name, exp in experiments.items():
    F = exp["res"].F

    rows.append({
        "Algoritmo": name,
        "HV ↑": hv_indicator(F),
        "IGD+ ↓": igdplus_indicator(F),
        "epsilon+ ↓": additive_epsilon(F, PF),
        "n_solutions": len(F),
        "tiempo_s": exp["time"]
    })

summary = pd.DataFrame(rows).set_index("Algoritmo")
summary.round(5)


## 10. Una mirada todavía más temprana

Ya comparamos 1200 y 2000 evaluaciones. Ahora retrocedemos aún más, aproximadamente al **25% del presupuesto**.

Esta vista sirve para recordar que, al principio, ninguno de los algoritmos está todavía cerca del Pareto front.


In [ ]:
def closest_snapshot(exp, target_eval):
    h = exp["history"]
    idx = int(np.argmin(np.abs(h["n_eval"].to_numpy() - target_eval)))
    return h.iloc[idx]["n_eval"], exp["snapshots"][idx]

target = int(0.25 * BUDGET)

for name, exp in experiments.items():
    n_eval, F_early = closest_snapshot(exp, target)

    plt.figure(figsize=(7, 5))
    plt.plot(PF[:, 0], PF[:, 1], "--", linewidth=2, label="Pareto front")
    plt.scatter(F_early[:, 0], F_early[:, 1], s=34, label=f"{name} @ {n_eval} eval.")
    plt.xlabel("$f_1$")
    plt.ylabel("$f_2$")
    plt.title(f"{name}: etapa temprana de la búsqueda")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


## 11. Pregunta de cierre del mini laboratorio

Compara:

1. la inspección visual,
2. la evolución de HV,
3. la evolución de IGD+,
4. la tabla a **1200 vs. 2000 evaluaciones**,
5. la tabla final con HV, IGD+ y $\epsilon^+$.

### ¿Qué conclusión publicarías?

Y, más importante:

### ¿Qué información adicional necesitarías antes de afirmar que un algoritmo es superior?

> **Una sola corrida es una demostración, no evidencia experimental suficiente.**  
> En un estudio formal deberíamos usar múltiples semillas, resumir la distribución de resultados y aplicar un análisis estadístico apropiado.

Dos recordatorios:

- **el indicador elegido puede cambiar la conclusión**;
- **el presupuesto de evaluaciones también puede cambiarla**.


## 12. Para experimentar después del taller

Cambia **una sola cosa**:

```python
SEED = 21
```

y vuelve a ejecutar desde la configuración.

Pregúntate:

- ¿se mantiene el ranking?
- ¿se mantienen las diferencias?
- ¿alguna métrica cambia de conclusión?

Después puedes probar un presupuesto distinto o aumentar el tamaño de población.


In [ ]:
# Guardar resultados de esta ejecución.
# Esto nos servirá para preparar un "plan B" antes del taller.

summary.round(8).to_csv("ECE2026_minilab_resumen.csv")
budget_table.round(8).to_csv("ECE2026_comparacion_presupuestos.csv")
winner_table.to_csv("ECE2026_ganadores_por_presupuesto.csv")

for name, exp in experiments.items():
    safe_name = name.lower().replace("-", "_")
    exp["history"].to_csv(f"ECE2026_historia_{safe_name}.csv", index=False)
    np.save(f"ECE2026_frente_final_{safe_name}.npy", exp["res"].F)

print("Resultados guardados en el entorno de Colab.")
print("\nTABLA FINAL:")
display(summary.round(5))
print("\nCOMPARACIÓN POR PRESUPUESTO:")
display(budget_table.round(5))
print("\nGANADORES POR PRESUPUESTO:")
display(winner_table)
